# NLP Final: TF-IDF lost-item candidate ranking

This notebook has two separate datasets: transcribed monthly inventory totals and a small, fictional matching benchmark. **Only the fictional benchmark is used to train and score matching methods.** A match score suggests records for staff review; it cannot prove ownership or authorize release.

## Workflow

1. Define the lost-to-found ranking task and its human-review boundary.
2. Record sources; profile aggregate inventory and prepare controlled examples.
3. Validate columns, IDs, labels, missing values and grouped data splits.
4. Compare a Logistic Regression baseline with TF-IDF and other rankers.
5. Tune on validation data; report held-out accuracy and ranking measures.
6. Inspect explanations and demo the service and Android app.

## 1. Transcribed inventory totals

`data/raw/campus/office_inventory_aggregate.csv` contains 57 category rows from the user-provided June–July and August 2026 summaries: 388 and 299 items. These are aggregate counts, not item reports or match labels, so they describe inventory only. Original names are kept beside normalized labels. The June–July umbrella disposal field is blank in the source and remains unknown. Verify the transcription with the office before submission.

In [1]:
import csv
from collections import defaultdict
from pathlib import Path
inventory_path = Path('data/raw/campus/office_inventory_aggregate.csv')
with inventory_path.open(encoding='utf-8', newline='') as stream:
    inventory = list(csv.DictReader(stream))
totals = defaultdict(int); category_totals = defaultdict(lambda: defaultdict(int))
for row in inventory:
    quantity = int(row['quantity'])
    totals[row['period']] += quantity
    category_totals[row['period']][row['category']] += quantity
print('period | rows | quantity | missing disposal destination')
for period in totals:
    rows = [row for row in inventory if row['period'] == period]
    missing_destination = sum(not row['dispose_to'].strip() for row in rows)
    print(f"{period} | {len(rows)} | {totals[period]} | {missing_destination}")
    print('  category totals:', dict(category_totals[period]))
print('Used for training or scoring: no. The table has no item-level match labels.')

period | rows | quantity | missing disposal destination
2026-06_07 | 33 | 388 | 1
  category totals: {'electronics': 125, 'calculators': 23, 'clothing': 58, 'other': 182}
2026-08 | 24 | 299 | 0
  category totals: {'electronics': 95, 'calculators': 7, 'clothing': 56, 'other': 141}
Used for training or scoring: no. The table has no item-level match labels.


In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv, json
ROOT = Path.cwd()
assert (ROOT / 'scripts/prepare_ranking_benchmark.py').exists(), 'Run from repository root'
from scripts.prepare_ranking_benchmark import main as prepare
from scripts.validate_ranking_dataset import validate
_NOTEBOOK_TMP = TemporaryDirectory(prefix='nlp-ranking-')
RUN_DIR = Path(_NOTEBOOK_TMP.name)
DATA = RUN_DIR / 'ranking'
prepare(DATA)
audit = validate(DATA)
assert audit['valid'], audit['errors']
print(json.dumps(audit, indent=2))

{
  "inventory_quantities": {
    "2026-06_07": 388,
    "2026-08": 299
  },
  "found_records": 24,
  "lost_reports": 27,
  "controlled_positive_labels": 24,
  "controlled_negative_labels": 168,
  "no_match_queries": 3,
  "candidate_pool_size": 24,
  "source_type": "AI-authored fictional controlled examples",
  "review": "Independent student review pending; no genuine same-item labels or campus performance."
}
{
  "counts": {
    "found_reports.csv": 24,
    "lost_reports.csv": 27,
    "match_labels.csv": 192,
    "evaluation_queries.csv": 27
  },
  "label_counts": {
    "1": 24,
    "0": 168
  },
  "errors": [],
  "valid": true,
  "note": "Structural checks do not verify physical identity, independent authorship, or label truth."
}


## 2. Fictional matching benchmark

The preparation script creates found reports, lost claims, pair labels and evaluation queries. Its validator checks columns, IDs, references, labels and splits, but not real-world truth. Item groups never cross train, validation and test; the candidate pool has 24 fictional found reports. No-match queries have no relevant found ID.

In [3]:
def read_csv(name):
    with (DATA / name).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))
found = read_csv('found_reports.csv'); lost = read_csv('lost_reports.csv')
labels = read_csv('match_labels.csv'); queries = read_csv('evaluation_queries.csv')
for split in ('train', 'validation', 'test'):
    groups = {r['item_group_id'] for r in found if r['split'] == split}
    print(split, 'item groups:', len(groups), 'matched queries:', sum(q['split']==split and q['has_match']=='1' for q in queries))
print('sources:', sorted({r['source_type'] for r in found}))
print('example:', {k: found[0][k] for k in ('found_id','category','description','source_type')})

train item groups: 8 matched queries: 8
validation item groups: 8 matched queries: 8
test item groups: 8 matched queries: 8
sources: ['ai_constructed_fictional']
example: {'found_id': 'f-g001', 'category': 'plastic_water_bottle', 'description': 'Clear plastic bottle with orange flip lid', 'source_type': 'ai_constructed_fictional'}


In [4]:
from collections import Counter
from statistics import median

def inspect_table(name, rows, id_field):
    ids = [row.get(id_field, '').strip() for row in rows]
    descriptions = [row.get('description', '').strip() for row in rows]
    categories = Counter(row.get('category', '').strip() or '[missing]' for row in rows)
    print(f'{name}: {len(rows)} rows; duplicate {id_field}: {len(ids) - len(set(ids))}; blank descriptions: {sum(not text for text in descriptions)}')
    print(f'  description characters: min={min(map(len, descriptions))}, median={median(map(len, descriptions))}, max={max(map(len, descriptions))}')
    print('  category counts:', dict(sorted(categories.items())))

inspect_table('found', found, 'found_id')
inspect_table('lost', lost, 'lost_id')
print('pair-label counts:', dict(Counter(row['label'] for row in labels)))
print('query split counts:', dict(Counter(row['split'] for row in queries)))

found: 24 rows; duplicate found_id: 0; blank descriptions: 0
  description characters: min=34, median=42.5, max=54
  category counts: {'calculator': 2, 'headphones': 1, 'laptop_charger': 3, 'metallic_bottle': 3, 'mouse': 2, 'pencil_pouch': 3, 'plastic_water_bottle': 3, 'spectacle_case': 1, 'spectacles': 1, 'sweater': 1, 'umbrella': 1, 'usb_cable': 3}
lost: 27 rows; duplicate lost_id: 0; blank descriptions: 0
  description characters: min=28, median=54, max=67
  category counts: {'calculator': 2, 'headphones': 1, 'laptop_charger': 3, 'metallic_bottle': 3, 'mouse': 2, 'pencil_pouch': 3, 'plastic_water_bottle': 3, 'scarf': 1, 'spectacle_case': 1, 'spectacles': 1, 'sweater': 1, 'umbrella': 2, 'usb_cable': 3, 'watch': 1}
pair-label counts: {'1': 24, '0': 168}
query split counts: {'train': 8, 'validation': 9, 'test': 10}


## 3. Baseline: Logistic Regression

Fit a fresh classifier on engineered pair features from the 64 training pairs. This is a comparison model, not the model loaded by the live worker. All results remain synthetic.

In [5]:
from matching_system.logistic_regression_features import FEATURE_NAMES, pair_features
from scripts.evaluate_ranking_benchmark import as_report
train_labels = [r for r in labels if r['split']=='train']
found_by_id = {r['found_id']: r for r in found}; lost_by_id = {r['lost_id']: r for r in lost}
example = train_labels[0]
features = pair_features(as_report(lost_by_id[example['lost_id']], 'lost'), as_report(found_by_id[example['found_id']], 'found'))
print('training pairs:', len(train_labels), 'features:', list(features))

training pairs: 64 features: ['description_similarity', 'category_match', 'category_conflict', 'colour_match', 'colour_conflict', 'brand_match', 'brand_conflict', 'model_match', 'model_conflict', 'material_match', 'material_conflict', 'distinctive_feature_similarity', 'location_similarity', 'time_difference_hours']


## 4. Final method: TF-IDF retrieval

TF-IDF emphasizes words that distinguish reports; unigrams and bigrams capture words and short phrases. Fit vocabulary on training text only. Cosine similarity measures text closeness, not ownership probability.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from matching_system.nlp_retrieval import normalize_report_text
training_text = [normalize_report_text(r['description']) for r in found + lost if r['split']=='train']
vectorizer = TfidfVectorizer(ngram_range=(1,2), strip_accents='unicode').fit(training_text)
query_text = normalize_report_text(lost_by_id['l-g017']['description'])
found_text = normalize_report_text(found_by_id['f-g017']['description'])
similarity = cosine_similarity(vectorizer.transform([query_text]), vectorizer.transform([found_text]))[0,0]
print('vocabulary size:', len(vectorizer.vocabulary_), 'example cosine:', round(float(similarity), 4))

vocabulary size: 205 example cosine: 0.1682


## 5. Validation and model selection

Compare keyword overlap, structured attributes, Logistic Regression, TF-IDF and a hybrid on the same candidate pool. Choose the hybrid weight and no-match cutoff on validation data. Keep the test split for final reporting only.

In [7]:
from scripts.evaluate_ranking_benchmark import main as evaluate
results = evaluate(data_dir=DATA, result_path=RUN_DIR/'ranking_benchmark.json')
print('selected TF-IDF weight:', results['hybrid']['tfidf_weight_selected_on_validation'])
print('training feature count:', len(results['logistic_regression']['features']))

{
  "keyword": {
    "matched_queries": 8,
    "no_match_queries": 2,
    "candidate_pool_size": 24,
    "top1_accuracy": 0.25,
    "recall_at_5": 0.875,
    "mrr": 0.540625,
    "matched_details": [
      {
        "query_id": "q-g017",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g018",
        "first_rank": 8,
        "recall_at_5": 0.0
      },
      {
        "query_id": "q-g019",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g020",
        "first_rank": 5,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g021",
        "first_rank": 2,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g022",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g023",
        "first_rank": 1,
        "recall_at_5": 1.0
      },
      {
        "query_id": "q-g024",
        "first_rank": 2,
        "recall_at_5": 1.0
      }
    ],
  

## 6. Held-out results

Top-1 accuracy counts matched queries ranked first. Recall@5 counts queries with a correct item in the first five. MRR rewards higher ranks. False suggestions count no-match queries whose top score passes a validation-set cutoff. Denominators are printed. These are fictional benchmark scores, not campus recovery rates.

In [8]:
print('method | Top-1 accuracy (matched queries) | Recall@5 | MRR | no-match false suggestions')
for name, item in results['methods'].items():
    test = item['test']
    print(f"{name} | {test['top1_accuracy']:.3f} ({test['matched_queries']}) | {test['recall_at_5']:.3f} | {test['mrr']:.3f} | {item['test_no_match_false_suggestions']}/{item['test_no_match_denominator']}")

method | Top-1 accuracy (matched queries) | Recall@5 | MRR | no-match false suggestions
keyword | 0.250 (8) | 0.875 | 0.541 | 1/2
attributes | 1.000 (8) | 1.000 | 1.000 | 1/2
logistic_regression | 1.000 (8) | 1.000 | 1.000 | 1/2
tfidf | 0.250 (8) | 0.500 | 0.391 | 2/2
hybrid | 1.000 (8) | 1.000 | 1.000 | 2/2


## 7. Explanation and input checks

The preview returns text similarity, fact statuses and staff guidance. This in-process check exercises explanation and validation logic only; it does not test HTTP, Firebase or Android. Similarity is not a probability.

In [9]:
from pydantic import ValidationError
from matching_system.api import PreviewRequest, preview_matches
request = PreviewRequest(item_name='Metal water bottle',description='Blue bottle with a scratched lid',category_id='metal_water_bottle',location_id='msb',event_date='2026-11-10',limit=3)
body = preview_matches(request)
print('status:', body['status'], 'first ID:', body['matches'][0]['found_report_id'])
print('explanation:', body['matches'][0]['explanation'])
for label, values in [('blank description', {'item_name':'Water bottle','description':''}), ('invalid calendar date', {'item_name':'Water bottle','description':'Blue bottle','event_date':'2026-02-30'})]:
    try:
        PreviewRequest(**values)
    except ValidationError:
        print(label + ': rejected')

status: possible_matches first ID: demo-found-metal
explanation: Facts: category agrees; location differs; event date agrees. Staff should review the conflicting details. Text similarity is lexical evidence, not a probability. Ownership is unverified; release is not authorized.
blank description: rejected
invalid calendar date: rejected


## 8. Optional live HTTP check

Start the local API with `python -m uvicorn matching_system.api:app --host 127.0.0.1 --port 8000`, then set `NLP_LIVE_HTTP_DEMO=1` before running this cell. It checks valid text, blank text and an invalid date. The local preview is not the authenticated Android workflow; see the Android repository's presentation guide for that demo.

In [10]:
import os
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def post_preview(payload):
    req = Request('http://127.0.0.1:8000/preview/matches', data=json.dumps(payload).encode('utf-8'), headers={'Content-Type':'application/json'}, method='POST')
    try:
        with urlopen(req, timeout=5) as response:
            return response.status, json.loads(response.read().decode('utf-8'))
    except HTTPError as error:
        return error.code, json.loads(error.read().decode('utf-8'))

if os.environ.get('NLP_LIVE_HTTP_DEMO') == '1':
    valid = {'item_name':'Metal water bottle','description':'Blue bottle with a scratched lid','category_id':'metal_water_bottle','location_id':'msb','event_date':'2026-11-10','limit':3}
    status, result = post_preview(valid)
    print('valid request:', status, result['status'], 'candidate count:', len(result['matches']))
    print('first explanation:', result['matches'][0]['explanation'] if result['matches'] else 'No candidate in the demo inventory')
    print('blank description:', post_preview({**valid, 'description':''})[0])
    print('invalid date:', post_preview({**valid, 'event_date':'2026-02-30'})[0])
else:
    print('Live HTTP check skipped. Start the local API and set NLP_LIVE_HTTP_DEMO=1 to run it.')

Live HTTP check skipped. Start the local API and set NLP_LIVE_HTTP_DEMO=1 to run it.


## 9. Interpretation and limits

The structured and hybrid methods score perfectly on eight fictional matched test queries because the generator reuses category and colour fields. TF-IDF misses paraphrases and flags both fictional no-match queries. These tiny constructed results reveal benchmark weaknesses; they are not real-world rates.

Before claiming campus performance, collect permitted, independently verified item pairs and keep a separate holdout. The team must verify the aggregate transcription, review synthetic examples and labels, run the Android demo and approve the notebook.